# 08 受控流程对比：只替换音高与节拍模块

同一段约 20 秒的项目内录音上比较：

| 流程 | 音高 | 节拍 | 和弦 | 对照目的 |
|:---|:---|:---|:---|:---|
| A | pYIN | librosa.beat | 共享模板分支 | 非学习型音高/节拍 |
| B | torchcrepe | Beat This! | 共享模板分支 | 学习型音高/节拍 |

两条 MIDI 使用完全相同的模板和弦铺底，以隔离音高与节拍模块的输出差异。

## 1. 环境自检与配置

In [ ]:
import sys
import subprocess
assert sys.version_info >= (3, 9), "需要 Python 3.9+"

import numpy as np
import matplotlib.pyplot as plt

# 配置 matplotlib 中文字体（macOS/Windows/Linux）
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['PingFang HK', 'STHeiti', 'Heiti TC', 'Arial Unicode MS', 'Hiragino Sans GB', 'Microsoft YaHei', 'SimHei', 'Noto Sans CJK SC', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False  # 解决负号显示为方框的问题

from IPython import display as ipydisplay
from pathlib import Path
import warnings

import librosa
import librosa.display

SAMPLE_RATE = 22050

# 路径推断：从 cwd 向上找含 CODE/datasets 的目录
_p = Path.cwd()
while not (_p / "CODE" / "datasets").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/datasets 的目录），请在项目内运行本 Notebook")
    _p = _parent
BASE_DIR = _p
DATASET_DIR = BASE_DIR / "CODE" / "datasets" / "audio_author"
OUTPUT_FIG_DIR = BASE_DIR / "CODE" / "chapter05" / "output_figures"
OUTPUT_AUDIO_DIR = BASE_DIR / "CODE" / "chapter05" / "output_audio"
OUTPUT_FIG_DIR.mkdir(exist_ok=True)
OUTPUT_AUDIO_DIR.mkdir(exist_ok=True)

print(f"librosa={librosa.__version__}")

## 2. 加载目标音频

In [ ]:
# 选取人声歌曲片段做无真值模块输出对照
AUDIO_PATH = DATASET_DIR / "xiaohetang_full.wav"
START_SEC = 30.0
DURATION_SEC = 20.0

audio_samples, sr = librosa.load(
    AUDIO_PATH, sr=SAMPLE_RATE, mono=True, offset=START_SEC, duration=DURATION_SEC
)
time_axis = np.linspace(START_SEC, START_SEC + len(audio_samples)/sr, len(audio_samples), endpoint=False)

print(f"加载音频：{AUDIO_PATH.name}")
print(f"切片：{START_SEC:.1f}s – {START_SEC + DURATION_SEC:.1f}s，共 {len(audio_samples)} 采样点")
display(ipydisplay.Audio(audio_samples, rate=SAMPLE_RATE))

## 3. 共享输入与流程 A

流程 A 使用 pYIN 与 librosa 节拍跟踪器。和弦模板分支是 A/B 共享控制变量，包含相似度时间平滑和低能量 `N` 标签。


In [ ]:
# A1：音高（pYIN）
f0_pyin, voiced_flag, _ = librosa.pyin(
    audio_samples,
    fmin=librosa.note_to_hz("C3"),
    fmax=librosa.note_to_hz("C6"),
    sr=SAMPLE_RATE
)
time_pyin = librosa.times_like(f0_pyin, sr=SAMPLE_RATE)

# 只保留 voiced 帧
f0_pyin_clean = np.where(voiced_flag, f0_pyin, np.nan)

print(f"流程 A—音高：pYIN 输出 {len(f0_pyin)} 帧，其中 voiced 帧 {np.sum(voiced_flag)} 个")
valid_pyin = f0_pyin_clean[np.isfinite(f0_pyin_clean)]
if valid_pyin.size:
    print(f"音高范围：{valid_pyin.min():.1f} – {valid_pyin.max():.1f} Hz")
else:
    print("音高范围：没有有限的 voiced F0 输出")

In [ ]:
# A2：节拍
tempo_a, beat_frames_a = librosa.beat.beat_track(y=audio_samples, sr=SAMPLE_RATE, hop_length=512)
beat_times_a = librosa.frames_to_time(beat_frames_a, sr=SAMPLE_RATE, hop_length=512)

tempo_a_value = float(np.asarray(tempo_a).reshape(-1)[0])
print(f"流程 A—节拍：估计速度 ≈ {tempo_a_value:.1f} BPM，检测到 {len(beat_times_a)} 个节拍")

In [ ]:
# 共享和弦分支：24 个 maj/min 模板 + 相似度平滑 + N 门限
from scipy.ndimage import median_filter

def generate_chord_templates():
    templates = {}
    names = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
    for root in range(12):
        major = np.zeros(12)
        major[(root + np.array([0, 4, 7])) % 12] = 1.0
        templates[f"{names[root]}:maj"] = major / np.linalg.norm(major)

        minor = np.zeros(12)
        minor[(root + np.array([0, 3, 7])) % 12] = 1.0
        templates[f"{names[root]}:min"] = minor / np.linalg.norm(minor)
    return templates

def chord_recognition_template(chroma, rms, templates, smooth_size=9, silence_ratio=0.02):
    template_names = list(templates)
    template_matrix = np.array([templates[name] for name in template_names])
    chroma_magnitude = np.linalg.norm(chroma, axis=0)
    valid_chroma = chroma_magnitude > 0
    chroma_norm = np.divide(
        chroma, chroma_magnitude[None, :],
        out=np.zeros_like(chroma),
        where=valid_chroma[None, :],
    )
    similarities = template_matrix @ chroma_norm
    similarities = median_filter(similarities, size=(1, smooth_size), mode="nearest")

    n_frames = min(chroma.shape[1], len(rms))
    similarities = similarities[:, :n_frames]
    best_idx = np.argmax(similarities, axis=0)
    best_scores = np.max(similarities, axis=0)
    threshold = silence_ratio * np.max(rms) if np.max(rms) > 0 else 0.0
    no_chord = (rms[:n_frames] <= threshold) | ~valid_chroma[:n_frames]
    labels = [
        "N" if no_chord[t] else template_names[index]
        for t, index in enumerate(best_idx)
    ]
    return labels, best_scores, similarities, no_chord

chroma_shared = librosa.feature.chroma_cqt(
    y=audio_samples, sr=SAMPLE_RATE, hop_length=512
)
rms_shared = librosa.feature.rms(
    y=audio_samples, frame_length=2048, hop_length=512
)[0]
templates = generate_chord_templates()
chords_shared, chord_scores_shared, chord_sims_shared, no_chord_shared = chord_recognition_template(
    chroma_shared, rms_shared, templates
)
time_chords_shared = librosa.frames_to_time(
    np.arange(len(chords_shared)), sr=SAMPLE_RATE, hop_length=512
)

print(f"共享和弦分支：{len(set(chords_shared))} 种标签，N 帧 {np.sum(no_chord_shared)}")
print("前 20 帧：", chords_shared[:20])


## 4. 流程 B：torchcrepe + Beat This!

流程 B 只替换音高与节拍模块；和弦仍使用上面的共享模板结果。依赖未预装时明确跳过，不自动安装。


In [ ]:
# 学习型依赖状态
torchcrepe_available = False
beat_this_available = False

try:
    import torch
    import torchcrepe as tc
    torchcrepe_available = True
    print("torchcrepe: 可用（默认 Viterbi decoder）")
except ImportError:
    print("torchcrepe: 未安装")

try:
    import warnings
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore", message=r".*torch\.cuda\.amp\.autocast.*", category=FutureWarning,
        )
        from beat_this.inference import File2Beats
    beat_this_available = True
    print("Beat This!: 可用")
except ImportError:
    print("Beat This!: 未安装")


In [ ]:
# 受控设计：两条流程共享相同和弦标签与时间轴
chords_a = chords_shared
time_chords_a = time_chords_shared
chords_b = chords_shared
time_chords_b = time_chords_shared

print("A/B 和弦分支对象一致：", chords_a is chords_b)


In [ ]:
f0_crepe_clean = time_crepe = periodicity = None
if torchcrepe_available:
    audio_tensor = torch.from_numpy(audio_samples).unsqueeze(0).float()
    # torchcrepe 会在离散音高 bin 上加入随机音分抖动；固定并恢复 NumPy 状态以稳定图表。
    dither_rng_state = np.random.get_state()
    np.random.seed(202605)
    try:
        f0_crepe, periodicity = tc.predict(
            audio_tensor, SAMPLE_RATE, hop_length=256,
            fmin=librosa.note_to_hz("C3"), fmax=librosa.note_to_hz("C6"),
            model="tiny", return_periodicity=True, device="cpu"
        )
    finally:
        np.random.set_state(dither_rng_state)
    f0_crepe = f0_crepe.squeeze().cpu().numpy()
    periodicity = periodicity.squeeze().cpu().numpy()
    time_crepe = librosa.times_like(f0_crepe, sr=SAMPLE_RATE, hop_length=256)

    conf_mask = periodicity > 0.5
    f0_crepe_clean = np.where(conf_mask, f0_crepe, np.nan)
    valid_values = f0_crepe_clean[np.isfinite(f0_crepe_clean)]
    print(f"流程 B—音高：{len(f0_crepe)} 帧，其中高置信度帧 {np.sum(conf_mask)} 个")
    if len(valid_values):
        print(f"高置信度范围：{valid_values.min():.1f} – {valid_values.max():.1f} Hz")
else:
    print("流程 B—音高：未运行")


In [ ]:
beat_times_b = np.array([])
if beat_this_available:
    import soundfile as sf
    temp_path = OUTPUT_AUDIO_DIR / "_temp_pipeline_b.wav"
    sf.write(temp_path, audio_samples, SAMPLE_RATE, subtype="PCM_16")
    try:
        file2beats = File2Beats(device="cpu")
        beat_times_b, downbeat_times_b = file2beats(str(temp_path))
        beat_times_b = np.asarray(beat_times_b, dtype=float)
        print(f"流程 B—节拍：{len(beat_times_b)} 个节拍，{len(downbeat_times_b)} 个小节首拍")
        print("前 5 个节拍：", np.round(beat_times_b[:5], 3))
    except Exception as exc:
        beat_times_b = np.array([])
        beat_this_available = False
        print(f"流程 B—节拍推理失败，未生成结果：{type(exc).__name__}: {exc}")
    finally:
        temp_path.unlink(missing_ok=True)
else:
    print("流程 B—节拍：未运行")


In [ ]:
# MIDI 可听化工具；A/B 使用相同和弦分支
try:
    import pretty_midi
    pretty_midi_available = True
except ImportError:
    pretty_midi_available = False
    print("pretty_midi 未安装，跳过 MIDI 生成。")

def f0_to_midi_notes(f0_hz, times, min_duration=0.05):
    """把连续 F0 量化到 12-TET MIDI，仅用于本 Notebook 的可听化。"""
    f0_hz = np.asarray(f0_hz, dtype=float).reshape(-1)
    times = np.asarray(times, dtype=float).reshape(-1)
    n_frames = min(len(f0_hz), len(times))
    if n_frames == 0:
        return []
    f0_hz, times = f0_hz[:n_frames], times[:n_frames]
    positive_steps = np.diff(times)
    positive_steps = positive_steps[positive_steps > 0]
    frame_step = float(np.median(positive_steps)) if positive_steps.size else min_duration
    final_end = float(times[-1] + frame_step)

    notes = []
    current_start = None
    current_pitch = None
    for t, freq in zip(times, f0_hz):
        valid = np.isfinite(freq) and freq > 0 and np.isfinite(t)
        midi_pitch = int(np.clip(np.round(librosa.hz_to_midi(freq)), 0, 127)) if valid else None
        if midi_pitch is None or (current_pitch is not None and midi_pitch != current_pitch):
            if current_start is not None and float(t) - current_start >= min_duration:
                notes.append((current_start, float(t), current_pitch))
            current_start = current_pitch = None
        if midi_pitch is not None and current_start is None:
            current_start, current_pitch = float(t), midi_pitch

    if current_start is not None and final_end - current_start >= min_duration:
        notes.append((current_start, final_end, current_pitch))
    return notes

def label_segments(labels, times):
    """把逐帧标签合并为不重叠的连续区间。"""
    labels = list(labels)
    times = np.asarray(times, dtype=float).reshape(-1)
    n_frames = min(len(labels), len(times))
    if n_frames == 0:
        return []
    labels, times = labels[:n_frames], times[:n_frames]
    positive_steps = np.diff(times)
    positive_steps = positive_steps[positive_steps > 0]
    frame_step = float(np.median(positive_steps)) if positive_steps.size else 0.2
    segments = []
    start = float(times[0])
    current = labels[0]
    for index in range(1, n_frames):
        if labels[index] != current:
            segments.append((start, float(times[index]), current))
            start, current = float(times[index]), labels[index]
    segments.append((start, float(times[-1] + frame_step), current))
    return segments

def create_midi_from_pipeline(pitch_hz, pitch_times, beat_times, chords, chord_times, tempo=120):
    if not pretty_midi_available:
        return None
    tempo = float(tempo) if np.isfinite(tempo) and tempo > 0 else 120.0
    pm = pretty_midi.PrettyMIDI(initial_tempo=tempo)

    melody = pretty_midi.Instrument(program=0)
    for start, end, pitch in f0_to_midi_notes(pitch_hz, pitch_times):
        melody.notes.append(pretty_midi.Note(
            velocity=80, pitch=int(pitch),
            start=float(start), end=float(end)
        ))
    pm.instruments.append(melody)

    drums = pretty_midi.Instrument(program=0, is_drum=True)
    for t in np.asarray(beat_times, dtype=float).reshape(-1):
        if not np.isfinite(t) or t < 0:
            continue
        drums.notes.append(pretty_midi.Note(
            velocity=100, pitch=35, start=float(t), end=float(t + 0.05)
        ))
    pm.instruments.append(drums)

    pad = pretty_midi.Instrument(program=49)
    offsets_by_quality = {"maj": [0, 4, 7], "min": [0, 3, 7]}
    for start, end, chord_label in label_segments(chords, chord_times):
        if chord_label == "N" or ":" not in chord_label:
            continue
        root_name, quality = chord_label.split(":", 1)
        offsets = offsets_by_quality.get(quality)
        if offsets is None:
            continue
        root_midi = librosa.note_to_midi(root_name + "4")
        for offset in offsets:
            pad.notes.append(pretty_midi.Note(
                velocity=60, pitch=int(np.clip(root_midi + offset, 0, 127)),
                start=float(start), end=float(end)
            ))
    pm.instruments.append(pad)
    return pm

def synthesize_with_click_track(midi_data, beat_times, sample_rate=SAMPLE_RATE):
    """用 pretty_midi 的简易合成器渲染旋律/和弦，并显式叠加可听节拍点击。"""
    # PrettyMIDI.synthesize 会忽略鼓轨；标准 MIDI 中仍保留鼓事件，WAV 另行叠加点击。
    synthesized = midi_data.synthesize(fs=sample_rate)
    if synthesized.size == 0:
        return synthesized
    beat_times = np.asarray(beat_times, dtype=float).reshape(-1)
    duration = len(synthesized) / sample_rate
    valid_times = beat_times[np.isfinite(beat_times) & (beat_times >= 0) & (beat_times < duration)]
    clicks = librosa.clicks(
        times=valid_times, sr=sample_rate, length=len(synthesized),
        click_freq=1000.0, click_duration=0.03,
    )
    mixed = synthesized + 0.3 * clicks
    peak = np.max(np.abs(mixed))
    return 0.95 * mixed / peak if peak > 0 else mixed


In [ ]:
# 明确记录控制变量：A/B 的和弦输入完全相同
assert chords_a is chords_b
assert time_chords_a is time_chords_b
print("控制变量确认：两条流程的和弦标签和时间轴完全共享。")


## 5. 并排可视化

上两行比较音高与节拍输出；下行只显示一次共享模板和弦，避免把控制变量误画成性能对比。

In [ ]:
plt.rcParams["font.sans-serif"] = ["PingFang HK", "Heiti TC", "Hiragino Sans GB", "Arial Unicode MS", "STHeiti", "Microsoft YaHei", "SimHei", "Noto Sans CJK SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(time_pyin, f0_pyin_clean, "o-", color="0.1", markersize=2, lw=0.8, label="A：pYIN")
if torchcrepe_available:
    axes[0].plot(time_crepe, f0_crepe_clean, "s-", color="0.65", markersize=2, lw=0.8, label="B：torchcrepe")
axes[0].set_ylabel("频率 (Hz)")
axes[0].set_title("音高输出（无真值）", loc="left")
axes[0].legend()
axes[0].set_ylim(0, 550)

librosa.display.waveshow(audio_samples, sr=SAMPLE_RATE, ax=axes[1], alpha=0.2, color="black")
axes[1].vlines(beat_times_a, -1, 1, color="0.15", ls="--", lw=1.3,
               label=f"A：librosa.beat（{len(beat_times_a)} 个）")
if beat_this_available:
    axes[1].vlines(beat_times_b, -1, 1, color="0.6", ls="-", lw=1.3,
                   label=f"B：Beat This!（{len(beat_times_b)} 个）")
axes[1].set_xlabel("")
axes[1].set_ylabel("振幅")
axes[1].set_title("节拍时间戳（无真值）", loc="left")
axes[1].legend(loc="upper center", bbox_to_anchor=(0.5, 1.17), ncol=2)

unique_chords = sorted(set(chords_shared))
chord_to_int = {label: index for index, label in enumerate(unique_chords)}
int_seq = np.array([chord_to_int[label] for label in chords_shared])
axes[2].step(time_chords_shared, int_seq, where="mid", color="0.3", lw=1.5,
             label="A/B 共享模板和弦")
axes[2].set_yticks(range(len(unique_chords)))
axes[2].set_yticklabels(unique_chords, fontsize=8)
axes[2].set_xlabel("时间 (s)")
axes[2].set_ylabel("模板标签")
axes[2].set_title("共享控制变量：相似度平滑 + N 门限", loc="left")
axes[2].legend()
axes[2].set_ylim(-0.5, len(unique_chords) - 0.5)

axes[0].set_xlim(0, DURATION_SEC)
plt.suptitle(
    f"受控流程对比：{AUDIO_PATH.name} [{START_SEC:.0f}s–{START_SEC+DURATION_SEC:.0f}s]",
    fontsize=14, x=0.02, ha="left"
)
plt.tight_layout()
plt.savefig(OUTPUT_FIG_DIR / "pipeline_comparison.png", dpi=600, bbox_inches="tight")
plt.show()


## 6. 共享和弦条件下的 MIDI 可听化

两条 MIDI 的和弦铺底完全相同，只让旋律轨迹与鼓点击时间不同。为了生成标准 MIDI，本节把连续 F0 四舍五入到最近的 12-TET MIDI 音高，并使用固定合成力度。`PrettyMIDI.synthesize` 的简易波形合成不渲染鼓轨，因此下方代码在生成 WAV 时按各自节拍时间戳显式叠加短点击，MIDI 文件中的标准鼓事件保持不变。这些量化、力度与点击音色设置只适用于当前可听化，听感差异仍不能替代带真值评测。

In [ ]:
midi_a = midi_b = None
if pretty_midi_available:
    tempo_value = float(np.asarray(tempo_a).reshape(-1)[0])
    midi_a = create_midi_from_pipeline(
        f0_pyin_clean, time_pyin, beat_times_a,
        chords_shared, time_chords_shared, tempo=tempo_value
    )
    midi_path_a = OUTPUT_AUDIO_DIR / "pipeline_a_output.mid"
    midi_a.write(str(midi_path_a))
    print(f"流程 A MIDI：{midi_path_a}")

    if torchcrepe_available and beat_this_available:
        midi_b = create_midi_from_pipeline(
            f0_crepe_clean, time_crepe, beat_times_b,
            chords_shared, time_chords_shared, tempo=tempo_value
        )
        midi_path_b = OUTPUT_AUDIO_DIR / "pipeline_b_output.mid"
        midi_b.write(str(midi_path_b))
        print(f"流程 B MIDI：{midi_path_b}")

    import soundfile as sf
    audio_synth_a = synthesize_with_click_track(midi_a, beat_times_a)
    wav_path_a = OUTPUT_AUDIO_DIR / "pipeline_a_synth.wav"
    sf.write(wav_path_a, audio_synth_a, SAMPLE_RATE)
    print(f"流程 A 合成音频：{wav_path_a}")

    if midi_b is not None:
        audio_synth_b = synthesize_with_click_track(midi_b, beat_times_b)
        wav_path_b = OUTPUT_AUDIO_DIR / "pipeline_b_synth.wav"
        sf.write(wav_path_b, audio_synth_b, SAMPLE_RATE)
        print(f"流程 B 合成音频：{wav_path_b}")
else:
    print("跳过 MIDI 生成")


In [ ]:
# 可选 FluidSynth 渲染；不影响核心实验
fluidsynth_available = False
try:
    result = subprocess.run(["fluidsynth", "--version"], capture_output=True, text=True)
    fluidsynth_available = result.returncode == 0
except Exception:
    pass

if fluidsynth_available and pretty_midi_available:
    # 优先项目自带 SoundFont，保证跨环境渲染一致；系统 FluidR3 作为备选
    sf_paths = [
        BASE_DIR / "CODE" / "datasets" / "soundfonts" / "TimGM6mb.sf2",
        "/usr/share/sounds/sf2/FluidR3_GM.sf2",
        "/opt/anaconda3/share/soundfonts/FluidR3_GM.sf2",
    ]
    sf_path = next((path for path in sf_paths if Path(path).exists()), None)
    midi_pairs = [("a", midi_path_a)]
    if midi_b is not None:
        midi_pairs.append(("b", midi_path_b))
    if sf_path:
        for name, midi_path in midi_pairs:
            out_wav = OUTPUT_AUDIO_DIR / f"pipeline_{name}_rendered.wav"
            subprocess.run(
                # fluidsynth 2.x 要求选项位于输入文件之前
                ["fluidsynth", "-ni", "-F", str(out_wav), "-r", str(SAMPLE_RATE),
                 sf_path, str(midi_path)],
                check=False, capture_output=True
            )
            if out_wav.exists():
                print(f"FluidSynth 渲染：{out_wav}")
            else:
                print(f"FluidSynth 渲染失败：{midi_path}")
    else:
        print("FluidSynth 可用，但未找到 SoundFont；跳过。")
else:
    print("FluidSynth 不可用或 pretty_midi 未安装，跳过。")


## 7. 小结与限制

1. A/B 只替换音高与节拍模块；模板和弦是共享控制变量。
2. 静音/低能量帧显式输出 `N`，相似度先沿时间平滑再 argmax。
3. BACHI 属于符号域模型，不参与本节的受控音频对照。
4. 当前片段没有逐帧真值：连续性、点数、音区或节拍点数量更多都不等于准确率。
5. 性能比较需要带 F0、节拍、和弦标注的任务域数据与对应指标。


In [ ]:
print("本 Notebook 生成的输出文件：")
for d in [OUTPUT_FIG_DIR, OUTPUT_AUDIO_DIR]:
    for f in sorted(d.glob("pipeline_*")):
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name:45s} {size_kb:8.1f} KB")